# Sequence-to-Sequence Neural Machine Translation

A clean, modern implementation of encoder-decoder NMT with TensorFlow/Keras.

**Architecture progression:**
1. Basic LSTM encoder-decoder
2. Bidirectional encoder
3. Attention mechanism

This notebook uses current Keras 3 best practices and proper masking throughout.

## Setup

In [3]:
import numpy as np
import tensorflow as tf
from pathlib import Path

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Load and Prepare Data

In [4]:
# Download Spanish-English dataset
url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
path = tf.keras.utils.get_file("spa-eng.zip", origin=url, extract=True)

# Find the extracted file
for file_path in Path(path).parent.rglob('spa.txt'):
    if file_path.is_file():
        text = file_path.read_text()
        break

# Clean and split into pairs
text = text.replace("¡", "").replace("¿", "")
pairs = [line.split("\t")[:2] for line in text.splitlines()]  # [:2] ignores attribution column
np.random.seed(42)
np.random.shuffle(pairs)

english_sentences, spanish_sentences = zip(*pairs)
print(f"Total sentence pairs: {len(english_sentences):,}")
print(f"\nExamples:")
for i in range(3):
    print(f"  {english_sentences[i]} => {spanish_sentences[i]}")

Total sentence pairs: 118,964

Examples:
  How boring! => Qué aburrimiento!
  I love sports. => Adoro el deporte.
  Would you like to swap jobs? => Te gustaría que intercambiemos los trabajos?


## Text Vectorization

We use special tokens for the decoder:
- `[START]` - signals the decoder to begin generating
- `[END]` - signals the decoder to stop generating

The decoder input is `[START] + sentence`, and the target is `sentence + [END]`.

In [5]:
# Hyperparameters
VOCAB_SIZE = 24000
MAX_LENGTH = 20
EMBED_DIM = 512
UNITS = 1024
STARTTOKEN = "starttoken"
ENDTOKEN = "endtoken"

# Train/validation split
TRAIN_SIZE = 100000

# Create vectorization layers
eng_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=MAX_LENGTH,
    standardize="lower_and_strip_punctuation",
    name="english_vectorizer"
)

spa_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=MAX_LENGTH,
    standardize="lower_and_strip_punctuation",
    name="spanish_vectorizer"
)

# Adapt on training data
eng_vectorizer.adapt(english_sentences[:TRAIN_SIZE])
spa_vectorizer.adapt([f"{STARTTOKEN} {s} {ENDTOKEN}" for s in spanish_sentences[:TRAIN_SIZE]])

# Check vocabularies
print("English vocab (first 10):", eng_vectorizer.get_vocabulary()[:10])
print("Spanish vocab (first 10):", spa_vectorizer.get_vocabulary()[:10])

I0000 00:00:1776269953.570169  273876 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21949 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


English vocab (first 10): ['', '[UNK]', 'the', 'i', 'to', 'you', 'tom', 'a', 'is', 'he']
Spanish vocab (first 10): ['', '[UNK]', 'starttoken', 'endtoken', 'de', 'que', 'a', 'no', 'tom', 'la']


In [ ]:
# Prepare datasets
# Encoder input: English sentence
# Decoder input: [START] + Spanish sentence
# Target: Spanish sentence + [END]

def prepare_data(eng_sentences, spa_sentences):
    encoder_inputs = tf.constant(eng_sentences)
    decoder_inputs = tf.constant([f"{STARTTOKEN} {s}" for s in spa_sentences])
    targets = spa_vectorizer([f"{s} {ENDTOKEN}" for s in spa_sentences])
    return encoder_inputs, decoder_inputs, targets

X_train_enc, X_train_dec, Y_train = prepare_data(
    english_sentences[:TRAIN_SIZE],
    spanish_sentences[:TRAIN_SIZE]
)
X_val_enc, X_val_dec, Y_val = prepare_data(
    english_sentences[TRAIN_SIZE:],
    spanish_sentences[TRAIN_SIZE:]
)

print(f"Training samples: {len(X_train_enc):,}")
print(f"Validation samples: {len(X_val_enc):,}")

Training samples: 100,000
Validation samples: 18,964


## Model 1: Basic Encoder-Decoder

The simplest seq2seq architecture:
- Encoder: LSTM that reads the source sentence and produces a context vector (final hidden state)
- Decoder: LSTM that generates the target sentence, initialized with the encoder's context

In [ ]:
def build_basic_seq2seq():
    # Inputs (strings)
    encoder_input = tf.keras.Input(shape=(), dtype=tf.string, name="encoder_input")
    decoder_input = tf.keras.Input(shape=(), dtype=tf.string, name="decoder_input")

    # Vectorize
    encoder_tokens = eng_vectorizer(encoder_input)
    decoder_tokens = spa_vectorizer(decoder_input)

    # Embeddings (mask_zero=True for proper padding handling)
    encoder_embedding = tf.keras.layers.Embedding(
        VOCAB_SIZE, EMBED_DIM, mask_zero=True, name="encoder_embedding"
    )(encoder_tokens)

    decoder_embedding = tf.keras.layers.Embedding(
        VOCAB_SIZE, EMBED_DIM, mask_zero=True, name="decoder_embedding"
    )(decoder_tokens)

    # Encoder: processes source, returns final states
    encoder_lstm = tf.keras.layers.LSTM(UNITS, return_state=True, name="encoder_lstm", dropout=0.25)
    _, state_h, state_c = encoder_lstm(encoder_embedding)
    encoder_states = [state_h, state_c]

    # Decoder: generates target, initialized with encoder states
    decoder_lstm = tf.keras.layers.LSTM(
        UNITS, return_sequences=True, name="decoder_lstm", dropout=0.25
    )
    decoder_output = decoder_lstm(decoder_embedding, initial_state=encoder_states)

    # Output projection
    output = tf.keras.layers.Dense(VOCAB_SIZE, activation="softmax", name="output")(decoder_output)

    model = tf.keras.Model([encoder_input, decoder_input], output, name="basic_seq2seq")
    return model

model_basic = build_basic_seq2seq()
model_basic.summary()

from tensorflow.keras.utils import plot_model

plot_model(
    model_basic,
    to_file="model_basic.png",
    show_shapes=True,           # shows tensor shapes at each layer
    show_layer_names=True,      # shows layer names
    show_layer_activations=True,# shows activation functions
    rankdir="TB",               # TB = top-to-bottom, LR = left-to-right
    dpi=150
)

NameError: name 'tf' is not defined

In [7]:
model_basic.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

history_basic = model_basic.fit(
    [X_train_enc, X_train_dec], Y_train,
    validation_data=([X_val_enc, X_val_dec], Y_val),
    batch_size=64,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint("model_basic.keras", save_best_only=True)
    ]
)

Epoch 1/10


I0000 00:00:1776268513.052194  272703 cuda_dnn.cc:529] Loaded cuDNN version 90300


 162/1563 ━━━━━━━━━━━━━━━━━━━━ 1:16 55ms/step - accuracy: 0.0521 - loss: 7.0551

KeyboardInterrupt: 

## Model 2: Bidirectional Encoder

Improvement: read the source sentence in both directions to capture context from both sides.

In [ ]:
def build_bidirectional_seq2seq():
    # Inputs
    encoder_input = tf.keras.Input(shape=(), dtype=tf.string, name="encoder_input")
    decoder_input = tf.keras.Input(shape=(), dtype=tf.string, name="decoder_input")

    # Vectorize
    encoder_tokens = eng_vectorizer(encoder_input)
    decoder_tokens = spa_vectorizer(decoder_input)

    # Embeddings
    encoder_embedding = tf.keras.layers.Embedding(
        VOCAB_SIZE, EMBED_DIM, mask_zero=True, name="encoder_embedding"
    )(encoder_tokens)

    decoder_embedding = tf.keras.layers.Embedding(
        VOCAB_SIZE, EMBED_DIM, mask_zero=True, name="decoder_embedding"
    )(decoder_tokens)

    # Bidirectional encoder
    encoder = tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(UNITS // 2, return_state=True, dropout=0.25),
        name="encoder_bilstm"
    )
    _, fwd_h, fwd_c, bwd_h, bwd_c = encoder(encoder_embedding)

    # Concatenate forward and backward states
    state_h = tf.keras.layers.Concatenate()([fwd_h, bwd_h])
    state_c = tf.keras.layers.Concatenate()([fwd_c, bwd_c])
    encoder_states = [state_h, state_c]

    # Decoder
    decoder_lstm = tf.keras.layers.LSTM(
        UNITS, return_sequences=True, name="decoder_lstm", dropout=0.25
    )
    decoder_output = decoder_lstm(decoder_embedding, initial_state=encoder_states)

    # Output
    output = tf.keras.layers.Dropout(0.3)(decoder_output)
    output = tf.keras.layers.Dense(VOCAB_SIZE, activation="softmax", name="output")(output)

    model = tf.keras.Model([encoder_input, decoder_input], output, name="bidirectional_seq2seq")
    return model

model_bidir = build_bidirectional_seq2seq()
model_bidir.summary()

In [ ]:
model_bidir.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

history_bidir = model_bidir.fit(
    [X_train_enc, X_train_dec], Y_train,
    validation_data=([X_val_enc, X_val_dec], Y_val),
    batch_size=64,
    epochs=15,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint("model_bidir.keras", save_best_only=True)
    ]
)

## Model 3: Attention

Key improvement: instead of compressing the entire source into a single context vector,
attention allows the decoder to look back at all encoder outputs and focus on relevant parts
at each decoding step.

In [7]:
def build_attention_seq2seq():
    # Inputs
    encoder_input = tf.keras.Input(shape=(), dtype=tf.string, name="encoder_input")
    decoder_input = tf.keras.Input(shape=(), dtype=tf.string, name="decoder_input")

    # Vectorize
    encoder_tokens = eng_vectorizer(encoder_input)
    decoder_tokens = spa_vectorizer(decoder_input)

    # Embeddings
    encoder_embedding = tf.keras.layers.Embedding(
        VOCAB_SIZE, EMBED_DIM, mask_zero=True, name="encoder_embedding"
    )(encoder_tokens)

    decoder_embedding = tf.keras.layers.Embedding(
        VOCAB_SIZE, EMBED_DIM, mask_zero=True, name="decoder_embedding"
    )(decoder_tokens)

    # Encoder returns full sequence (for attention) and final state (for decoder init)
    encoder_lstm = tf.keras.layers.LSTM(
        4*UNITS, return_sequences=True, return_state=True, name="encoder_lstm", dropout=0.2
    )
    encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

    # Decoder consumes target sequence and starts from encoder final state
    decoder_lstm = tf.keras.layers.LSTM(
        4*UNITS, return_sequences=True, name="decoder_lstm", dropout=0.2
    )
    decoder_outputs = decoder_lstm(decoder_embedding, initial_state=[state_h, state_c])

    # Drop masks before attention to avoid broadcast issues with symbolic masks in Keras 3
    encoder_outputs_nomask = tf.keras.layers.Lambda(
        lambda x: x, name="encoder_outputs_nomask"
    )(encoder_outputs)
    decoder_outputs_nomask = tf.keras.layers.Lambda(
        lambda x: x, name="decoder_outputs_nomask"
    )(decoder_outputs)

    # Cross-attention: queries=decoder, keys/values=encoder
    attention = tf.keras.layers.AdditiveAttention(name="cross_attention")
    context = attention([decoder_outputs_nomask, encoder_outputs_nomask])

    # Fuse decoder signal + context, then project to vocab logits
    combined = tf.keras.layers.Concatenate(name="attn_concat")(
        [decoder_outputs_nomask, context]
    )
    output = tf.keras.layers.Dropout(0.3)(combined)
    output = tf.keras.layers.Dense(UNITS, activation="tanh", name="projection")(output)
    output = tf.keras.layers.Dropout(0.3)(output)
    output = tf.keras.layers.Dense(VOCAB_SIZE, activation="softmax", name="output")(output)

    model = tf.keras.Model([encoder_input, decoder_input], output, name="attention_seq2seq")
    return model

model_attn = build_attention_seq2seq()
model_attn.summary()

/opt/tljh/user/lib/python3.12/site-packages/keras/src/layers/layer.py:939: UserWarning: Layer 'encoder_outputs_nomask' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/opt/tljh/user/lib/python3.12/site-packages/keras/src/layers/layer.py:939: UserWarning: Layer 'decoder_outputs_nomask' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "attention_seq2seq"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ english_vectorizer  │ (None, 20)        │          0 │ encoder_input[0]… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spanish_vectorizer  │ (None, 20)        │          0 │ decoder_input[0]… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 20, 512)   │ 12,288,000 │ english_vectoriz… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 20)        │          0 │ english_vectoriz… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 20, 512)   │ 12,288,000 │ spanish_vectoriz… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 20,       │ 75,513,856 │ encoder_embeddin… │
│                     │ 4096), (None,     │            │ not_equal[0][0]   │
│                     │ 4096), (None,     │            │                   │
│                     │ 4096)]            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ (None, 20, 4096)  │ 75,513,856 │ decoder_embeddin… │
│                     │                   │            │ encoder_lstm[0][… │
│                     │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 20)        │          0 │ spanish_vectoriz… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_outputs_no… │ (None, 20, 4096)  │          0 │ decoder_lstm[0][… │
│ (Lambda)            │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_outputs_no… │ (None, 20, 4096)  │          0 │ encoder_lstm[0][… │
│ (Lambda)            │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cross_attention     │ (None, 20, 4096)  │      4,096 │ decoder_outputs_… │
│ (AdditiveAttention) │                   │            │ encoder_outputs_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_concat         │ (None, 20, 8192)  │          0 │ decoder_outputs_… │
│ (Concatenate)       │                   │            │ cross_attention[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 20, 8192)  │          0 │ attn_concat[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ projection (Dense)  │ (None, 20, 1024)  │  8,389,632 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 208,597,440 (795.74 MB)

 Trainable params: 208,597,440 (795.74 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model_attn.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

history_attn = model_attn.fit(
    [X_train_enc, X_train_dec], Y_train,
    validation_data=([X_val_enc, X_val_dec], Y_val),
    batch_size=64,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint("model_attn.keras", save_best_only=True)
    ]
)

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 104s 66ms/step - accuracy: 0.0830 - loss: 5.5055 - val_accuracy: 0.1694 - val_loss: 3.0430
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 103s 66ms/step - accuracy: 0.1800 - loss: 2.7972 - val_accuracy: 0.2162 - val_loss: 2.1360
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 103s 66ms/step - accuracy: 0.2267 - loss: 1.8119 - val_accuracy: 0.2315 - val_loss: 1.8524
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 103s 66ms/step - accuracy: 0.2522 - loss: 1.2826 - val_accuracy: 0.2381 - val_loss: 1.7969
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 101s 65ms/step - accuracy: 0.2719 - loss: 0.9493 - val_accuracy: 0.2412 - val_loss: 1.8387
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 101s 65ms/step - accuracy: 0.2857 - loss: 0.7525 - val_accuracy: 0.2419 - val_loss: 1.9067


## Inference: Greedy Decoding

Generate translations one token at a time, always picking the most likely next token.

In [ ]:
def translate_greedy(model, sentence, max_length=MAX_LENGTH):
    """Translate using greedy decoding (always pick highest probability token)."""
    spa_vocab = spa_vectorizer.get_vocabulary()


    translation = STARTTOKEN

    for _ in range(max_length):
        # Get predictions
        enc_input = tf.constant([sentence])
        dec_input = tf.constant([translation])
        predictions = model.predict([enc_input, dec_input], verbose=0)

        # Get the last token's prediction
        next_token_probs = predictions[0, len(translation.split()) - 1]
        next_token_id = np.argmax(next_token_probs)
        next_token = spa_vocab[next_token_id]

        if next_token == ENDTOKEN:
            break

        translation += " " + next_token

    # Remove [START] token from output
    return translation.replace(STARTTOKEN, "").strip()

In [9]:
# Test translations
test_sentences = [
    "I like soccer.",
    "How are you?",
    "How are you, pal?",
    "The weather is nice today.",
    "I want to learn Spanish.",
    "Where is the library?",
    "We can order dinner but I don't want pizza.",
    "The only thing we have to fear is fear itself",
"When I fell on the football field I broke my leg"]

print("=" * 60)
print("Basic Model Translations:")
print("=" * 60)

model_basic = tf.keras.models.load_model("model_basic.keras")

for sent in test_sentences:
    translation = translate_greedy(model_basic, sent)
    print(f"EN: {sent}")
    print(f"ES: {translation}")
    print()


print("=" * 60)
print("Bidirectional Model Translations:")
print("=" * 60)

model_bidir = tf.keras.models.load_model("model_bidir.keras")

for sent in test_sentences:
    translation = translate_greedy(model_bidir, sent)
    print(f"EN: {sent}")
    print(f"ES: {translation}")
    print()

Basic Model Translations:


I0000 00:00:1776264393.566507  192883 cuda_dnn.cc:529] Loaded cuDNN version 90300


EN: I like soccer.
ES: me gusta el fútbol

EN: How are you?
ES: cómo están

EN: How are you, pal?
ES: cómo estáis usted

EN: The weather is nice today.
ES: el clima es hoy hoy

EN: I want to learn Spanish.
ES: quiero aprender español

EN: Where is the library?
ES: dónde está la biblioteca

EN: We can order dinner but I don't want pizza.
ES: puedo cenar pero no puedo jugar a la cena

EN: The only thing we have to fear is fear itself
ES: lo único que tenemos es esperar a la gente

EN: When I fell on the football field I broke my leg
ES: cuando salí me caí de la bicicleta me trenzaba la bicicleta

Bidirectional Model Translations:
EN: I like soccer.
ES: me gusta el fútbol

EN: How are you?
ES: cómo estás

EN: How are you, pal?
ES: cómo te estás escondiendo

EN: The weather is nice today.
ES: hoy hace buen tiempo

EN: I want to learn Spanish.
ES: quiero aprender chino

EN: Where is the library?
ES: dónde está la biblioteca

EN: We can order dinner but I don't want pizza.
ES: podemos invita

## Inference: Beam Search

Maintain multiple candidate translations and explore different paths.
Often produces better translations than greedy decoding.

In [ ]:
def translate_beam(model, sentence, beam_width=5, max_length=MAX_LENGTH):
    """Translate using beam search."""
    spa_vocab = spa_vectorizer.get_vocabulary()
    start_token = STARTTOKEN
    end_token = ENDTOKEN
    end_id = spa_vocab.index(end_token) if end_token in spa_vocab else 3

    # Initialize beam with start token
    # Each candidate is (log_probability, token_sequence)
    beam = [(0.0, start_token)]

    for _ in range(max_length):
        candidates = []

        for log_prob, sequence in beam:
            # Skip completed sequences
            if sequence.endswith(end_token):
                candidates.append((log_prob, sequence))
                continue

            # Get predictions
            enc_input = tf.constant([sentence])
            dec_input = tf.constant([sequence])
            predictions = model.predict([enc_input, dec_input], verbose=0)

            # Get probabilities for next token
            seq_len = len(sequence.split())
            next_probs = predictions[0, seq_len - 1]

            # Get top k candidates
            top_k_indices = np.argsort(next_probs)[-beam_width:]

            for idx in top_k_indices:
                token = spa_vocab[idx]
                new_log_prob = log_prob + np.log(next_probs[idx] + 1e-10)
                new_sequence = sequence + " " + token
                candidates.append((new_log_prob, new_sequence))

        # Keep top beam_width candidates
        beam = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]

        # Check if all beams have ended
        if all(seq.endswith(end_token) for _, seq in beam):
            break

    # Return best translation (remove special tokens)
    best_sequence = beam[0][1]
    result = best_sequence.replace(start_token, "").replace(end_token, "").strip()
    return result

In [15]:
# Compare greedy vs beam search
print("=" * 60)
print("Basic Model, Greedy vs Beam Search Comparison:")
print("=" * 60)

model_attn = tf.keras.models.load_model("model_attn.keras", safe_mode=False)
model = model_attn

for sent in test_sentences:
    greedy = translate_greedy(model, sent)
    beam = translate_beam(model, sent, beam_width=5)
    print(f"EN:     {sent}")
    print(f"Greedy: {greedy}")
    print(f"Beam:   {beam}")
    print()

Basic Model, Greedy vs Beam Search Comparison:
EN:     I like soccer.
Greedy: me gusta el fútbol
Beam:   me gusta el fútbol

EN:     How are you?
Greedy: cómo estás
Beam:   cómo estás

EN:     How are you, pal?
Greedy: cómo eres
Beam:   cómo eres

EN:     The weather is nice today.
Greedy: hoy hace buen tiempo
Beam:   hoy hace buen tiempo

EN:     I want to learn Spanish.
Greedy: quiero aprender español
Beam:   quiero aprender español

EN:     Where is the library?
Greedy: dónde está la biblioteca
Beam:   dónde está la biblioteca

EN:     We can order dinner but I don't want pizza.
Greedy: podemos ayudar pero no quiero cenar
Beam:   podemos cenar pero no quiero cocinar

EN:     The only thing we have to fear is fear itself
Greedy: la única cosa que se debe decir es inevitable
Beam:   la única cosa que se debe decir es inevitable

EN:     When I fell on the football field I broke my leg
Greedy: cuando entré al sol me rompí la pierna
Beam:   cuando entré al sol me rompí la pierna



## Compare Models

In [16]:
# Evaluate all models on validation set
print("Model Comparison (Validation Loss):")
print("-" * 40)

for name, model in [("Basic", model_basic), ("Bidirectional", model_bidir), ("Attention", model_attn)]:
    loss, acc = model.evaluate([X_val_enc, X_val_dec], Y_val, verbose=0)
    print(f"{name:15} - Loss: {loss:.4f}, Accuracy: {acc:.4f}")

Model Comparison (Validation Loss):
----------------------------------------
Basic           - Loss: 1.9473, Accuracy: 0.2245
Bidirectional   - Loss: 1.8811, Accuracy: 0.2303
Attention       - Loss: 1.8319, Accuracy: 0.2324


In [17]:
# Side-by-side translation comparison
print("\nTranslation Comparison:")
print("=" * 70)

for sent in test_sentences:
    print(f"English: {sent}")
    print(f"  Basic:       {translate_greedy(model_basic, sent)}")
    print(f"  Bidirect:    {translate_greedy(model_bidir, sent)}")
    print(f"  Attention:   {translate_greedy(model_attn, sent)}")
    print()


Translation Comparison:
English: I like soccer.
  Basic:       me gusta el fútbol
  Bidirect:    me gusta el fútbol
  Attention:   me gusta el fútbol

English: How are you?
  Basic:       cómo están
  Bidirect:    cómo estás
  Attention:   cómo estás

English: How are you, pal?
  Basic:       cómo estáis usted
  Bidirect:    cómo te estás escondiendo
  Attention:   cómo eres

English: The weather is nice today.
  Basic:       el clima es hoy hoy
  Bidirect:    hoy hace buen tiempo
  Attention:   hoy hace buen tiempo

English: I want to learn Spanish.
  Basic:       quiero aprender español
  Bidirect:    quiero aprender chino
  Attention:   quiero aprender español

English: Where is the library?
  Basic:       dónde está la biblioteca
  Bidirect:    dónde está la biblioteca
  Attention:   dónde está la biblioteca

English: We can order dinner but I don't want pizza.
  Basic:       puedo cenar pero no puedo jugar a la cena
  Bidirect:    podemos invitar a la cena pero no he almorzado
  

## Notes for Students

### Key Concepts

1. **Encoder-Decoder Architecture**: The encoder reads the source sequence and compresses it into a context representation. The decoder generates the target sequence from this context.

2. **Teacher Forcing**: During training, we feed the decoder the correct previous token (from the ground truth) rather than its own prediction. This speeds up training but can cause a mismatch with inference.

3. **Masking**: We use `mask_zero=True` on embeddings so the model ignores padding tokens (index 0). This mask propagates through LSTM and Attention layers.

4. **Attention**: Instead of compressing everything into a fixed vector, attention lets the decoder look at all encoder outputs and learn which parts are relevant for each output token.

5. **Beam Search**: Greedy decoding picks the best token at each step, but beam search maintains multiple hypotheses. This often finds better overall translations.

### Things to Try

- Increase/decrease `UNITS` and `EMBED_DIM`
- Add more LSTM layers (stacking)
- Try GRU instead of LSTM
- Experiment with different beam widths
- Add learning rate scheduling
- Implement nucleus sampling (top-p) for more diverse outputs